# Subfields de OpenAlex

In [1]:
import requests
import json

import pandas as pd

In [ ]:
results = []
for page in range(1, 4):
    url = f"https://api.openalex.org/subfields?page={page}&per_page=100"
    req = requests.get(url)
    if req.status_code == 200:
        data = req.json()
        results.extend(data["results"])
    else:
        print(f"Error: {req.status_code}")
        exit()
print(f"Total results fetched: {len(results)}")

In [ ]:
data_df = []
for result in results:
    new_row = {}
    url = result.get("id")
    iid = url.split("/")[-1]
    name = result.get("display_name")
    description = result.get("description")
    name_alternative = result.get("display_name_alternatives")
    field = result.get("field").get("display_name")

    data_df.append({"id": iid, "name": name, "description": description, 
    "name_alternative": name_alternative, "field": field, "url": url})

subfields_df = pd.DataFrame(data_df)
subfields_df.to_excel("subfields.xlsx", index=False)



In [15]:
len(results)

252

# Pipeline Honduras

con OpenAlex

Tipos búsqueda (type_Search):
- &search.title_and_abstract.exact= ... Búsqueda exacta
- &search.title_and_abstract= ... Búsqueda con derivaciones de las palabras (stemming)
- &search.title_abstract_keywords = ... Búsqueda con stemming incluyendo las palabras claves

Filtro de búsqueda
- &filter=publication_year:2019- Desde 2019 en adelante el año de publicación 

Selección de valores de salida
- &select=id,doi,display_name,...

Límites de consulta con API KEY
| Limit | Value |
|-------|-------|
| OR values per filter |	100 |
| per_page maximum | 100 |
| sample maximum | 10,000 |
| Basic paging limit | 10,000 results |

Se puede hacer más de 10,000 revisando la paginación de OpenAlex

In [1]:
import os
import time
import requests
import json
import pandas as pd

from dotenv import load_dotenv

In [2]:
load_dotenv()
OA_API_KEY = os.getenv("OPENALEX_API_KEY")

start_year = 2019

In [3]:
def build_openalex_url(query_search, start_year):
    url = f"https://api.openalex.org/works?api_key={OA_API_KEY}&\
filter=publication_year:{start_year}-&per_page=100\
&search.title_and_abstract.exact={query_search}"
    return url

In [4]:
query_search = "climate change"
build_openalex_url(start_year, query_search)

'https://api.openalex.org/works?api_key=loeJIEfoqaUSdyB9mcHCfY&filter=publication_year:climate change-&per_page=100&search.title_and_abstract.exact=2019'

## Prueba con búsquedas sin y con paginación

1.1 A
```
Honduras AND ("public services access" OR "acceso a servicios públicos" OR "quality of public services" OR "Calidad de servicios públicos" OR "equity in Access" OR "equidad en acceso" OR "electricity access" OR "acceso a la electricidad" OR "electricity services" OR "servicios eléctricos" OR "transport network" OR "red de transporte" OR "transportation system" OR "sistema de transporte" OR "urban transportation" OR "transporte urbano" OR "Acceso a electricidad" OR "access to electricity" OR "Electrificación rural" OR "rural electrification" OR "inclusive energy access" OR "Inclusión energética" OR "energy inclusion" OR "Movilidad urbana" OR "urban mobility")
```

1.2 - Requiere paginación
```
Honduras AND ("hospital services" OR "servicios hospitalarios" OR "health system" OR "sistema de salud" OR "public health" OR "salud pública" OR "education system" OR "sistema educativo" OR "rural education" OR "educación rural" OR "basic education" OR "educación básica" OR "education quality" OR "calidad educativa" OR "social protection" OR "protección social" OR "cash transfers" OR "transferencias monetarias" OR "vulnerable population" OR "población vulnerable" OR "indigenous people" OR "pueblos indígenas" OR "Capacidad hospitalaria" OR "hospital capacity" OR "Servicios de salud" OR "health services" OR "Acceso a educación" OR "access to education" OR "Calidad educativa" OR "educational quality" OR "Programas de protección social" OR "social protection programs" OR "Inclusión de población vulnerable" OR "inclusion of vulnerable populations")
```

En el doc de referencia no dice si usaron stemming o no.

In [5]:
search_a = {
    "query_search": 'Honduras AND ("public services access" OR "acceso a servicios públicos" OR "quality of public services" OR "Calidad de servicios públicos" OR "equity in Access" OR "equidad en acceso" OR "electricity access" OR "acceso a la electricidad" OR "electricity services" OR "servicios eléctricos" OR "transport network" OR "red de transporte" OR "transportation system" OR "sistema de transporte" OR "urban transportation" OR "transporte urbano" OR "Acceso a electricidad" OR "access to electricity" OR "Electrificación rural" OR "rural electrification" OR "inclusive energy access" OR "Inclusión energética" OR "energy inclusion" OR "Movilidad urbana" OR "urban mobility")',
    "pillar": "Upgrade public service delivery",
    "strategic_objective": "Strengthen the quality and access of public services"
}

search_b = {
    "query_search": 'Honduras AND ("hospital services" OR "servicios hospitalarios" OR "health system" OR "sistema de salud" OR "public health" OR "salud pública" OR "education system" OR "sistema educativo" OR "rural education" OR "educación rural" OR "basic education" OR "educación básica" OR "education quality" OR "calidad educativa" OR "social protection" OR "protección social" OR "cash transfers" OR "transferencias monetarias" OR "vulnerable population" OR "población vulnerable" OR "indigenous people" OR "pueblos indígenas" OR "Capacidad hospitalaria" OR "hospital capacity" OR "Servicios de salud" OR "health services" OR "Acceso a educación" OR "access to education" OR "Calidad educativa" OR "educational quality" OR "Programas de protección social" OR "social protection programs" OR "Inclusión de población vulnerable" OR "inclusion of vulnerable populations")',
    "pillar": "Upgrade public service delivery",
    "strategic_objective": "Reinforce human capital and social protection, with a focus on vulnerable populations"
}

In [ ]:
def build_openalex_url(query_search, start_year, type_search="title_abstract_keywords", per_page=100, cursor="*"):
    url = f"https://api.openalex.org/works?api_key={OA_API_KEY}&\
filter=publication_year:{start_year}-&per_page={per_page}&cursor={cursor}\
&search.{type_search}={query_search}"
    return url

def openalex_request(query_search, start_year, type_search="title_abstract_keywords", per_page=100):
    ''' Request to Open Alex API'''
    url = build_openalex_url(query_search, start_year, type_search, per_page)
    req = requests.get(url)
    if req.status_code == 200:
        data = req.json()
        total_cost = data['meta']['cost_usd']
        total_time_ms = data['meta']['db_response_time_ms']
        if data['meta']['count'] > per_page:
            if data['meta']['count'] > 10_000:
                print("Warning: Query surpass limit of 10.000 results")

            # Apply pagination
            print(f'Query surpass {per_page} results, apply pagination')
            cursor_i = data['meta']['next_cursor']
            page = 1
            results_extended = data['results']

            while cursor_i != None and page * per_page < 10_000:
                url = build_openalex_url(query_search, start_year, per_page, cursor_i)
                req = requests.get(url)
                data_pag = req.json()
                if req.status_code != 200:
                    break # Stop if any page breaks
                results_extended.extend(data_pag['results'])

                total_cost += data_pag['meta']['cost_usd']
                total_time_ms += data_pag['meta']['db_response_time_ms']
                cursor_i = data_pag['meta']['next_cursor']
                page += 1
                time.sleep(0.5)

            data['meta']['pages'] = page - 1
            data['meta']['db_response_time_ms'] = total_time_ms
            data['meta']['cost_usd'] = total_cost
            data['results'] = results_extended
        data['meta']['pages'] = 1
        return data
    else:
        print(f"Error: {req.status_code}")
        return None

In [7]:
openalex_data = openalex_request(search_a["query_search"], start_year)

In [8]:
print(
json.dumps(openalex_data['meta'], indent=4)
)

{
    "count": 24,
    "db_response_time_ms": 2125,
    "page": null,
    "per_page": 100,
    "next_cursor": null,
    "groups_count": null,
    "x_query": {
        "oql": "works where year >= (2019)\n  and title/abstract has (\n    \"Honduras\"\n    and (\n      \"Acceso a electricidad\" or \"Calidad de servicios p\u00fablicos\"\n      or \"Electrificaci\u00f3n rural\" or \"Inclusi\u00f3n energ\u00e9tica\" or \"Movilidad urbana\"\n      or \"acceso a la electricidad\" or \"acceso a servicios p\u00fablicos\"\n      or \"access to electricity\" or \"electricity access\"\n      or \"electricity services\" or \"energy inclusion\" or \"equidad en acceso\"\n      or \"equity in Access\" or \"inclusive energy access\"\n      or \"public services access\" or \"quality of public services\"\n      or \"red de transporte\" or \"rural electrification\"\n      or \"servicios el\u00e9ctricos\" or \"sistema de transporte\"\n      or \"transport network\" or \"transportation system\" or \"transport

In [9]:
len(openalex_data['results'])

24

In [10]:
print(
json.dumps(openalex_data['results'][0], indent=4)
)

{
    "id": "https://openalex.org/W4402339143",
    "doi": "https://doi.org/10.1016/j.engappai.2024.109249",
    "title": "A geospatial clustering algorithm and its integration into a techno-economic rural electrification planning model",
    "display_name": "A geospatial clustering algorithm and its integration into a techno-economic rural electrification planning model",
    "relevance_score": 51.610695,
    "publication_year": 2024,
    "publication_date": "2024-09-08",
    "ids": {
        "openalex": "https://openalex.org/W4402339143",
        "doi": "https://doi.org/10.1016/j.engappai.2024.109249"
    },
    "language": "en",
    "primary_location": {
        "id": "doi:10.1016/j.engappai.2024.109249",
        "is_oa": true,
        "landing_page_url": "https://doi.org/10.1016/j.engappai.2024.109249",
        "pdf_url": null,
        "source": {
            "id": "https://openalex.org/S900972176",
            "display_name": "Engineering Applications of Artificial Intelligence",


In [11]:
interest_vars = ["id", "doi", "display_name", "relevance_score",
                 "publication_year", "publication_date", "language", 
                 "primary_location.landing_page_url",
                 "primary_location.source.id", "primary_location.source.host_organization_name",
                 "primary_location.source.display_name", "primary_location.raw_source_name", 
                 "primary_location.source.type", "primary_location.raw_type", "type",
                 "primary_location.is_oa", "open_access.oa_status", "open_access.oa_url",
                 "authorships", # Is a list of authorship objects
                 "countries_distinct_count", "institutions_distinct_count", # To check authorships objects
                 "fwci", "cited_by_count", "citation_normalized_percentile.value",
                 "primary_topic.id", "primary_topic.display_name", 
                 "primary_topic.subfield.id", "primary_topic.subfield.display_name",
                 "primary_topic.field.id", "primary_topic.field.display_name",
                 "primary_topic.domain.id", "primary_topic.domain.display_name",
                 "updated_date"
]

# Inside "authorships" list
authors_vars = ["author.id", "author.display_name"]
# Inside "authorships[n].institutions" list
institutions_vars = ["id", "display_name", "country_code", "type"]

abstract_object = "abstract_inverted_index", # Abstract as set of words with list of their positions

# "keywords" is a list of keyword objects, each with "id" and "display_name". Get if asked for
# "sustainable_development_goals" is a list of SDG objects, each with "id" and "display_name". Get if asked for

In [12]:
print(len(interest_vars))

33


In [13]:
# Crear df con datos de interés
studies = pd.json_normalize(openalex_data['results'])[interest_vars]

In [14]:
studies.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 33 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              24 non-null     str    
 1   doi                                             23 non-null     str    
 2   display_name                                    24 non-null     str    
 3   relevance_score                                 24 non-null     float64
 4   publication_year                                24 non-null     int64  
 5   publication_date                                24 non-null     str    
 6   language                                        21 non-null     str    
 7   primary_location.landing_page_url               24 non-null     str    
 8   primary_location.source.id                      17 non-null     str    
 9   primary_location.source.host_organization_name  12 non-n

In [15]:
# Build authorships information
for au_var in authors_vars:
    studies[f'authorships.{au_var}'] = studies['authorships'].apply(
        lambda x: [pd.json_normalize(author)[au_var][0] for author in x] if isinstance(x, list) else []
        )
    studies[f'authorships.{au_var}'] = studies[f'authorships.{au_var}'].apply(lambda x: ';'.join(map(str, x)))

In [ ]:
# Build institutions information
studies['authorships.institutions'] = studies['authorships'].apply(
        lambda x: [author['institutions'] for author in x] if isinstance(x, list) else []
        )
for inst_var in institutions_vars:
    studies[f'authorships.institutions.{inst_var}'] = studies['authorships.institutions'].apply(
    lambda x: set([inst[inst_var] for sublist in x for inst in sublist]) if isinstance(x, list) else set()
    )
    studies[f'authorships.institutions.{inst_var}'] = studies[f'authorships.institutions.{inst_var}'].apply(lambda x: ';'.join(map(str, x)))


# Hay que hacer "display_name" - "country_code" por nombres comunes entre paises

In [17]:
# Build abstract/description
def build_abstract(abstract_inverted_index):
    if isinstance(abstract_inverted_index, dict):
        index_items= abstract_inverted_index.items()
        index_expand = {i: k for k, v in index_items for i in v}
        index_expand = sorted(index_expand.items(), key=lambda x: x[0])
        abstract = ' '.join([word for position, word in index_expand])

        # Filter printable chars
        abstract =  ''.join(ch for ch in abstract if ch.isprintable() or ch.isspace())
        return abstract
    else:
        return None

In [20]:
print(build_abstract(openalex_data['results'][0]["abstract_inverted_index"]))

Rural electrification planning is a complex process requiring careful consideration of various factors to ensure efficient and cost-effective solutions. Existing clustering methods in academic literature often fall short in this context, as they typically do not account for geographical barriers, restricted areas, and key electrical and geospatial metrics simultaneously. This can result in clusters that do not meet the energy needs of the study region, potentially causing inefficient energy distribution and increased costs. This study presents a novel clustering algorithm , RElect_MGEC (Rural Electrification Microgrid and Grid Extension Clustering), specifically designed for techno-economic planning in rural areas. The RElect_MGEC algorithm combines density-based and graph clustering methods to group households while considering constraints imposed by geographic barriers, electricity power, and distance from the generation center. The algorithm was implemented within the IntiGIS (Geogr

In [ ]:
studies['abstract_inverted_index'] = [result["abstract_inverted_index"] for result in openalex_data['results']]
studies['abstract'] = studies['abstract_inverted_index'].apply(build_abstract)

In [19]:
# Clean  objects
studies.drop(columns=['authorships', 'authorships.institutions', 'abstract_inverted_index'], inplace=True)

## Todas queries

Se utilizan todas las búsquedas de los 3 pilares definidos en el reporte bibliométrico de Honduras 2019 - 2026. En el reporte bibliométrico estuvieron 1425 resultados.

Las búsquedas se definen en el archivo ```data/queries_honduras_OE.json```.

In [ ]:
with open("../data/queries_honduras_OE.json", "r") as file:
    queries_honduras = json.load(file)

print(queries_honduras['meta'])

{'country': 'Honduras', 'start_year': 2019, 'end_year': 2026, 'type_search': 'exact'}


In [22]:
cost = 0
studies_df = pd.DataFrame()
print('Total queries:', len(queries_honduras['queries']))
for query in queries_honduras['queries']:
    print(f"Query {query['query_id']}: {query['query_search']}")
    q_data = openalex_request(query['query_search'], start_year)
    cost += q_data['meta']['cost_usd']

    with open(f"honduras_backup/query_{query['query_id']}.json", "w") as f: # Save backup
        f.write(json.dumps(q_data, indent=4))

    # Create dataframe -----
    studies = pd.json_normalize(q_data['results'])[interest_vars]

    # Build authorships information
    for au_var in authors_vars:
        studies[f'authorships.{au_var}'] = studies['authorships'].apply(
            lambda x: [pd.json_normalize(author)[au_var][0] for author in x] if isinstance(x, list) else []
            )
        studies[f'authorships.{au_var}'] = studies[f'authorships.{au_var}'].apply(lambda x: ';'.join(map(str, x)))

    # Build institutions information
    studies['authorships.institutions'] = studies['authorships'].apply(
            lambda x: [author['institutions'] for author in x] if isinstance(x, list) else []
            )
    for inst_var in institutions_vars:
        studies[f'authorships.institutions.{inst_var}'] = studies['authorships.institutions'].apply(
            lambda x: set([inst[inst_var] for sublist in x for inst in sublist]) if isinstance(x, list) else set()
            )
        studies[f'authorships.institutions.{inst_var}'] = studies[f'authorships.institutions.{inst_var}'].apply(lambda x: ';'.join(map(str, x)))

    # Build abstract/description
    studies['abstract_inverted_index'] = [result["abstract_inverted_index"] for result in q_data['results']]
    studies['abstract'] = studies['abstract_inverted_index'].apply(build_abstract)

    # Clean  objects
    studies.drop(columns=['authorships', 'authorships.institutions', 'abstract_inverted_index'], inplace=True)

    # Queries variables
    studies['pillar'] = query['pillar']
    studies['strategic_objective'] = query['strategic_objective']

    # Append
    studies_df = pd.concat([studies_df, studies], ignore_index=True)
    time.sleep(0.5)

print('Total OpenAlex cost USD:', round(cost, 4))

Total queries: 8
Query 1.1A: Honduras AND ("public services access" OR "acceso a servicios pÃºblicos" OR "quality of public services" OR "Calidad de servicios pÃºblicos" OR "equity in Access" OR "equidad en acceso" OR "electricity access" OR "acceso a la electricidad" OR "electricity services" OR "servicios elÃ©ctricos" OR "transport network" OR "red de transporte" OR "transportation system" OR "sistema de transporte" OR "urban transportation" OR "transporte urbano" OR "Acceso a electricidad" OR "access to electricity" OR "ElectrificaciÃ³n rural" OR "rural electrification" OR "inclusive energy access" OR "InclusiÃ³n energÃ©tica" OR "energy inclusion" OR "Movilidad urbana" OR "urban mobility")
Query 1.1B: Honduras AND ("service infrastructure" OR "infraestructura de servicios" OR "coverage of services" OR "cobertura de servicios" OR "water and sanitation" OR "agua y saneamiento" OR "sewer network" OR "red de alcantarillado" OR "Acceso a agua potable" OR "access to drinking water" OR "Ag

In [24]:
studies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1439 entries, 0 to 1438
Data columns (total 41 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              1439 non-null   str    
 1   doi                                             1248 non-null   str    
 2   display_name                                    1439 non-null   str    
 3   relevance_score                                 1439 non-null   float64
 4   publication_year                                1439 non-null   int64  
 5   publication_date                                1439 non-null   str    
 6   language                                        1285 non-null   str    
 7   primary_location.landing_page_url               1439 non-null   str    
 8   primary_location.source.id                      1226 non-null   str    
 9   primary_location.source.host_organization_name  892 

### Filtrar por topics/subfield
A diferencia del filtro por topics del documento usaré la clasificación por subfield debido a que es más fácil de estandarizar usando el archivo subfields_tagged.xlsx

In [7]:
import pandas as pd

subfields_df = pd.read_excel("subfields_tagged.xlsx")

In [8]:
subfields_df.head(3)

,id,name,description,name_alternative,field,url,subfield_honduras,interest_subfield
0,1100,General Agricultural and Biological Sciences,cultivation of plants and animals to provide u...,['plant agriculture'],Agricultural and Biological Sciences,https://openalex.org/subfields/1100,General Agricultural and Biological Sciences,1
1,1102,Agronomy and Crop Science,science and technology of producing and using ...,"['agriculture', 'agriculture (general)', 'scie...",Agricultural and Biological Sciences,https://openalex.org/subfields/1102,NaN,0
2,1103,Animal Science and Zoology,scientific study of animals,"['animal biology', 'animal science', 'zoologic...",Agricultural and Biological Sciences,https://openalex.org/subfields/1103,NaN,0


In [9]:
# Join ids
studies_df['primary_topic.subfield.id'] = studies_df['primary_topic.subfield.id'].str.split("/").str[-1].astype(str)
subfields_df = subfields_df.rename(columns={"id": "primary_topic.subfield.id"})
subfields_df['primary_topic.subfield.id'] = subfields_df['primary_topic.subfield.id'].astype(str)

In [10]:
studies_df = pd.merge(studies_df, subfields_df[['primary_topic.subfield.id', 'interest_subfield']], 
                       how="left", on ="primary_topic.subfield.id")

In [11]:
studies_df['interest_subfield'].value_counts(dropna=False)

interest_subfield
1.0    1113
0.0     217
NaN     109
Name: count, dtype: int64

In [ ]:
#studies_df.to_excel("openalex_studies_test.xlsx", index=False)

### Filtro de pertinencia

En texto: 
"revisión manual de las publicaciones restantes. La pertinencia se evaluó considerando el puntaje de relevancia asignado por OpenAlex (columna AL), así como el título, el resumen, y el tópico de OpenAlex"

Preguntas para Fernanda: 
- ¿cómo se define este umbral de relevancia? 
- ¿Complementa el filtro por subfield?
- Desde la búsqueda en OA puedo usar el filtro de tipo (type = article, book, conference-paper, report, etc
)

In [ ]:
import pandas as pd 
import numpy as np

In [64]:
studies_df = pd.read_excel("openalex_studies_test.xlsx")

In [65]:
studies_df['interest_subfield'].value_counts(dropna=False)

interest_subfield
1.0    1113
0.0     217
NaN     109
Name: count, dtype: int64

In [66]:
studies_df_filtered = studies_df[studies_df['interest_subfield'].isin([1, np.nan])].copy()

In [67]:
studies_df_filtered.shape

(1222, 42)

In [68]:
studies_df['type'].value_counts(dropna=False)

type
article                860
report                  96
preprint                89
other                   72
conference-paper        68
book                    56
book-chapter            56
dissertation            50
dataset                 42
conference-abstract     17
editorial               11
reference-entry          4
book-review              4
peer-review              4
paratext                 4
review                   3
letter                   2
erratum                  1
Name: count, dtype: int64

In [69]:
not_allowed_types = ["preprint", "review", "paratext", "editoral", "reference-entry", "letter", "erratum", "other"]
studies_df_filtered = studies_df_filtered[~studies_df_filtered['type'].isin(not_allowed_types)]

In [70]:
studies_df_filtered.shape

(1066, 42)

In [71]:
studies_df['relevance_score'].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])

count    1439.000000
mean        9.330354
std        16.249187
min         0.056144
5%          0.398209
10%         0.553724
25%         0.908213
50%         4.263700
75%         7.566562
90%        25.750211
95%        40.687077
max       164.769260
Name: relevance_score, dtype: float64

### Duplicados

En texto: "Las publicaciones duplicadas o que aparecían en más de una búsqueda (pueden haber resultado de varias búsquedas por palabras clave correspondientes a distintos objetivos estratégicos) fueron eliminadas, conservando únicamente el registro que indicaba el objetivo estratégico más pertinente según OpenAlex"

En los datos compartidos no está la columna AL (debe ser AK) que referencian. ¿Este ranking que usan sale del ```relevance_score```?

Implementar pero levantar la pregunta.

In [72]:
studies_df['doi'].duplicated().sum()

np.int64(280)

In [73]:
studies_df_filtered['doi'].duplicated().sum()

np.int64(206)

In [74]:
studies_df['display_name'].duplicated().sum()

np.int64(188)

In [75]:
studies_df_filtered['display_name'].duplicated().sum()

np.int64(119)

In [76]:
studies_df['relevance_score'].isna().sum()

np.int64(0)

In [77]:
studies_df_filtered['doi'].isna().sum()

np.int64(144)

In [79]:
for dup_var in ['doi', 'display_name']:
    temp = studies_df_filtered[studies_df_filtered[dup_var].isna()].copy()
    studies_df_filtered = studies_df_filtered[~studies_df_filtered[dup_var].isna()].copy()
    studies_df_filtered = studies_df_filtered.sort_values('relevance_score', ascending=False).\
        drop_duplicates(dup_var, keep='first').sort_index()
    studies_df_filtered = pd.concat([studies_df_filtered, temp])

In [81]:
studies_df_filtered.info()

<class 'pandas.DataFrame'>
Index: 946 entries, 3 to 1438
Data columns (total 42 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              946 non-null    str    
 1   doi                                             809 non-null    str    
 2   display_name                                    946 non-null    str    
 3   relevance_score                                 946 non-null    float64
 4   publication_year                                946 non-null    int64  
 5   publication_date                                946 non-null    str    
 6   language                                        858 non-null    str    
 7   primary_location.landing_page_url               946 non-null    str    
 8   primary_location.source.id                      815 non-null    str    
 9   primary_location.source.host_organization_name  583 non-nu

### Eliminación de caracteres especiales

En el texto se pone un listado. 

Eliminar patrones de html.


En los datos de muestra obtenidos desde la API no se encuentran 

### Imputación

Hablan solo de los doi en blanco, pero revisar otros

Preguntas para Fernanda:
- doi: Revisar más casos
- ¿Imputar lenguaje con el título? La descripción y abstract no debe usarse porque puede estar generado por OpenAlex
- primary_location.source.host_organization_name: No me parece que requiera imputación, es muy agregado

In [44]:
studies_df.isna().sum()

id                                                  0
doi                                               191
display_name                                        0
relevance_score                                     0
publication_year                                    0
publication_date                                    0
language                                          154
primary_location.landing_page_url                   0
primary_location.source.id                        213
primary_location.source.host_organization_name    547
primary_location.source.display_name              213
primary_location.raw_source_name                  331
primary_location.source.type                      213
primary_location.raw_type                          53
type                                                0
primary_location.is_oa                              0
open_access.oa_status                               0
open_access.oa_url                                330
countries_distinct_count    

### Limpieza de instituciones

Aquí es donde usan IA para normalizar y clasificar en:

- Honduras (o país X)
- LAC (excl. Honduras o país X)
- Estados Unidos
- Europa
- Organismos multilaterales
- Otras instituciones

¿Qué se hace cuando hay instituciones de múltiples continentes? ¿Se clasifica por cada institución o se agrupan todas las instituciones?

In [ ]:
inst_list = studies_df['authorships.institutions.display_name'].str.split(";").to_list()
inst_list = [i for l in inst_list for i in l if i != ""]
inst_list = set(inst_list)

print(len(inst_list), "instituciones únicas (texto sin normalizar)")

1118 instituciones únicas (sin normalizar)


In [72]:
inst_list

{'Universidad a Distancia de Madrid',
 'Universidad Europea de Madrid',
 'United Nations University World Institute for Development Economics Research',
 'University of New Haven',
 'Universidad Autónoma de Chiapas',
 'Hospital San Borja Arriarán',
 'Chaudhary Charan Singh Haryana Agricultural University',
 'Central University',
 'Texas A&M University – Central Texas',
 'University of Colorado Boulder',
 'Nippon Koei (Japan)',
 'Niroo Research Institute',
 'Heinrich Heine University Düsseldorf',
 'Ghana Health Service',
 'Hospital General San Juan de Dios',
 'Universidad del Rosario',
 'Universidade Católica do Salvador',
 'Operation PAR',
 'University of the Americas',
 'University of Oxford',
 'Instituto Politécnico Nacional',
 'Hospital Provincial de Rosario',
 'Dirección de Investigación y Desarrollo',
 'Duke Institute for Health Innovation',
 'Purdue University West Lafayette',
 'Secretariat of Public Education',
 'ET Enterprises (United Kingdom)',
 'University of Monterrey',
 'Ci

Por id

In [74]:
inst_list_id = studies_df['authorships.institutions.id'].str.split(";").to_list()
inst_list_id = [i for l in inst_list_id for i in l if i != ""]
inst_list_id = set(inst_list_id)

print(len(inst_list_id), "instituciones únicas (id)")

1139 instituciones únicas (id)


Por nombre se deben concatenar el nombre y el país, a causa de nombres comunes como 'Ministry of Economics'. Ajustarlo desde la construcción del display name al leer